# Generating OpenSpace Fieldline JSON & OSFLS from NOAA GFS

## Setup

Run this section before running the other sections.

In [ ]:
# Google Colab
!apt-get update -qq
!apt-get install -y libeccodes-dev
!pip install -q cfgrib eccodes xarray numpy matplotlib
# For running locally
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import json
import struct
import os
import urllib.request
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import torch
import torch.nn.functional as F

# Automatically detect GPU if available, fallback to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(torch.get_num_threads())
print(f"PyTorch using device: {DEVICE}")


R_EARTH = 6371000.0  # Earth radius in meters
ALTITUDE = 10000.0   # Altitude offset above surface in meters for fieldlines
# Set integration step time (e.g., 1800 seconds = 30 minutes of transport per step)
DT_SECONDS = 3600.0
NUM_STEPS = 40
NUM_FLOWLINES = 9000


def iso_to_j2000_seconds(iso_str):
    """Converts ISO timestamp string to J2000 seconds for OpenSpace time conversion."""
    # Format: YYYY-MM-DDThh:mm:ss.000
    cleaned_iso = iso_str.replace("-", "").replace(":", "").replace("T", " ")
    dt = datetime.strptime(cleaned_iso[:15], "%Y%m%d %H%M%S").replace(tzinfo=timezone.utc)

    # J2000 Epoch: 2000-01-01 12:00:00 UTC
    j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
    return (dt - j2000_epoch).total_seconds()

def latlon_to_cartesian(lat_deg, lon_deg, alt_m=ALTITUDE):
    """Converts Lat/Lon/Alt to Earth-Centered Earth-Fixed (ECEF) X,Y,Z in meters."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    r = R_EARTH + alt_m
    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)
    return [float(x), float(y), float(z)]

def trace_wind_fieldlines(ds, device=DEVICE):
    """
    GPU-accelerated streamline tracer using PyTorch bilinear grid sampling.
    Traces all NUM_FLOWLINES simultaneously in parallel.
    """
    # Extract variable names and data
    u_var = [v for v in ds.data_vars if 'u10' in v.lower() or 'u' in v.lower()][0]
    v_var = [v for v in ds.data_vars if 'v10' in v.lower() or 'v' in v.lower()][0]

    lat_grid = ds.latitude.values
    lon_grid = ds.longitude.values
    u_data = ds[u_var].values
    v_data = ds[v_var].values

    # GFS latitudes descending (90 -> -90)
    if lat_grid[0] > lat_grid[-1]:
        lat_grid = lat_grid[::-1]
        u_data = np.flip(u_data, axis=0)
        v_data = np.flip(v_data, axis=0)

    lat_min, lat_max = float(lat_grid[0]), float(lat_grid[-1])
    lon_min, lon_max = float(lon_grid[0]), float(lon_grid[-1])

    # Convert grid matrices to PyTorch Tensors on CUDA
    # Shape for grid_sample: (batch=1, channels=2, height, width)
    uv_tensor = torch.from_numpy(
        np.stack([u_data, v_data], axis=0)
    ).unsqueeze(0).float().to(device)

    # Equal-area sine sampling seed points
    np.random.seed(42)
    init_lats = np.degrees(np.arcsin(np.random.uniform(-0.995, 0.995, NUM_FLOWLINES)))
    init_lons = np.random.uniform(0, 360, NUM_FLOWLINES)

    curr_lats = torch.tensor(init_lats, dtype=torch.float32, device=device)
    curr_lons = torch.tensor(init_lons, dtype=torch.float32, device=device)

    # Storage array for output: list of lines, each containing list of [x, y, z, speed]
    # Shape: (NUM_STEPS, NUM_FLOWLINES, 4)
    all_steps_pts = []

    # Trigonometric constants for coordinate projection
    r_sphere = R_EARTH + ALTITUDE

    for step in range(NUM_STEPS):
        # Cartesian ECEF Conversion (Parallel GPU math)
        lat_rad = torch.deg2rad(curr_lats)
        lon_rad = torch.deg2rad(curr_lons)

        cos_lat = torch.cos(lat_rad)
        x = r_sphere * cos_lat * torch.cos(lon_rad)
        y = r_sphere * cos_lat * torch.sin(lon_rad)
        z = r_sphere * torch.sin(lat_rad)

        # Normalize coordinates to PyTorch grid_sample range [-1, 1]
        # Grid sample expects x = longitude normalized, y = latitude normalized
        norm_lon = 2.0 * (curr_lons - lon_min) / (lon_max - lon_min) - 1.0
        norm_lat = 2.0 * (curr_lats - lat_min) / (lat_max - lat_min) - 1.0

        # Shape: (1, 1, NUM_FLOWLINES, 2)
        grid_coords = torch.stack([norm_lon, norm_lat], dim=-1).unsqueeze(0).unsqueeze(0)

        # Sample U and V vectors simultaneously
        uv_sampled = F.grid_sample(uv_tensor, grid_coords, mode='bilinear', padding_mode='zeros', align_corners=True)
        # Shape: (NUM_FLOWLINES,)
        u_vals = uv_sampled[0, 0, 0, :]
        v_vals = uv_sampled[0, 1, 0, :]

        speeds = torch.sqrt(u_vals**2 + v_vals**2)

        # Pack current step positions [X, Y, Z, Speed]
        step_data = torch.stack([x, y, z, speeds], dim=-1)
        all_steps_pts.append(step_data)

        # Euler Advection Step
        d_lat = (v_vals * DT_SECONDS) / 111000.0
        safe_cos = torch.clamp(torch.abs(cos_lat), min=0.01)
        d_lon = (u_vals * DT_SECONDS) / (111000.0 * safe_cos)

        curr_lats = torch.clamp(curr_lats + d_lat, min=-85.0, max=85.0)
        curr_lons = torch.remainder(curr_lons + d_lon, 360.0)

    # Stack results back to CPU for file export
    # Stack shape: (NUM_STEPS, NUM_FLOWLINES, 4) -> Transpose to (NUM_FLOWLINES, NUM_STEPS, 4)
    flowlines_tensor = torch.stack(all_steps_pts, dim=0).permute(1, 0, 2).cpu().numpy()

    # Convert to standard Python lists for export_to_osfls compatibility
    fieldlines = [line.tolist() for line in flowlines_tensor]
    return fieldlines

def export_to_osfls(osfls_path, fieldlines, trigger_time_j2000, scalar_name="grid_value"):
    """Writes fieldlines directly to native OpenSpace OSFLS binary format."""
    n_lines = len(fieldlines)
    extra_names_bytes = scalar_name.encode('utf-8') + b'\0'
    byte_size_all_names = len(extra_names_bytes)

    line_start = []
    line_count = []
    vertex_positions = []
    extra_quantities = []

    curr_index = 0
    for line in fieldlines:
        pts_count = len(line)
        line_start.append(curr_index)
        line_count.append(pts_count)
        curr_index += pts_count

        for pt in line:
            # pt structure: [x, y, z, speed]
            vertex_positions.extend(pt[:3])
            extra_quantities.append(pt[3])

    n_points = curr_index

    # Convert arrays to standard C data types matching OpenSpace C++ structs
    line_start_arr = np.array(line_start, dtype=np.int32)
    line_count_arr = np.array(line_count, dtype=np.uint32)
    vertex_positions_arr = np.array(vertex_positions, dtype=np.float32)
    extra_quantities_arr = np.array(extra_quantities, dtype=np.float32)

    with open(osfls_path, 'wb') as ofs:
        # Header Fields
        ofs.write(struct.pack('i', 0))                    # CurrentVersion = 0
        ofs.write(struct.pack('d', trigger_time_j2000))    # _triggerTime
        ofs.write(struct.pack('i', 0))                    # _model = Invalid/Generic (0)
        ofs.write(struct.pack('B', 0))                    # _isMorphable = 0 (false)
        ofs.write(struct.pack('Q', n_lines))              # nLines
        ofs.write(struct.pack('Q', n_points))             # nPoints
        ofs.write(struct.pack('Q', 1))                    # nExtras (1 quantity: grid_value)
        ofs.write(struct.pack('Q', byte_size_all_names))  # byteSizeAllNames

        # Array Data Payload
        line_start_arr.tofile(ofs)
        line_count_arr.tofile(ofs)
        vertex_positions_arr.tofile(ofs)
        extra_quantities_arr.tofile(ofs)

        # String Data Payload
        ofs.write(extra_names_bytes)

## Color Table

In [ ]:
def generate_openspace_transfer_function(cmap_name="turbo", num_samples=10, output_file="wind-speed.txt"):
    """
    Generates a transfer function text file in OpenSpace mappingkey format.
    """
    cmap = plt.get_cmap(cmap_name)
    normalized_positions = np.linspace(0.0, 1.0, num_samples)

    with open(output_file, "w") as f:
        # Header setup
        f.write(f"width {num_samples}\n")
        f.write("lower 0.0\n")
        f.write("upper 1.0\n\n")

        # Keyframe mapping
        for pos in normalized_positions:
            # Matplotlib returns normalized RGBA floats (0.0 to 1.0)
            r, g, b, a = cmap(pos)

            # Convert RGBA to 0-255 integers
            r_int = int(round(r * 255))
            g_int = int(round(g * 255))
            b_int = int(round(b * 255))
            a_int = int(round(a * 255))

            f.write(f"mappingkey  {pos:.3f}     {r_int}   {g_int}    {b_int}   {a_int}\n")

    print(f"Transfer function successfully written to {output_file}")

generate_openspace_transfer_function("turbo", 10, "wind-speed.txt")

## Globe Grid to OS Fieldline JSON Demo

In [ ]:
def geodetic_to_ecef(lat_deg, lon_deg, alt_m):
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    r = R_EARTH + alt_m

    x = r * np.cos(lat_rad) * np.cos(lon_rad)
    y = r * np.cos(lat_rad) * np.sin(lon_rad)
    z = r * np.sin(lat_rad)
    return [float(x), float(y), float(z)]

def generate_lat_lon_grid(alt_m=10000, lat_step=15, lon_step=15, num_pts_per_line=72):
    """
    Generates a 3D wireframe grid at a specified altitude.
    """
    grid_lines = []

    # 1. Latitude Rings (Parallels)
    lats = np.arange(-75, 90, lat_step)
    lons_ring = np.linspace(-180, 180, num_pts_per_line)

    for lat in lats:
        ring_pts = []
        for lon in lons_ring:
            xyz = geodetic_to_ecef(lat, lon, alt_m)
            # [x, y, z, dummy_scalar]
            ring_pts.append(xyz + [1.0])
        grid_lines.append(ring_pts)

    # 2. Longitude Meridians
    lons = np.arange(-180, 180, lon_step)
    lats_meridian = np.linspace(-89, 89, num_pts_per_line)

    for lon in lons:
        meridian_pts = []
        for lat in lats_meridian:
            xyz = geodetic_to_ecef(lat, lon, alt_m)
            meridian_pts.append(xyz + [1.0])
        grid_lines.append(meridian_pts)

    return grid_lines

def write_grid_json(filename, grid_lines, dummy_time="2000-01-01T00:00:00.000"):
    output_dict = {}
    columns = ["x", "y", "z", "grid_value"]

    for idx, line in enumerate(grid_lines):
        output_dict[str(idx)] = {
            "time": dummy_time,
            "trace": {
                "columns": columns,
                "data": line
            }
        }

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output_dict, f, indent=2)

    print(f"Generated grid JSON with {len(grid_lines)} grid lines to {filename}")

if __name__ == "__main__":
    # Generate grid lines at 10 km altitude
    grid = generate_lat_lon_grid(alt_m=10000, lat_step=15, lon_step=15)
    write_grid_json("2026-09-23T12-00-00-000.json", grid)

## NOAA GFS to JSON

Code that I played around with until I figured out how to make OSFLS files.

### Get Current Timestamp

In [ ]:
# Calculate the most recent available 6-hour GFS cycle
now_utc = datetime.now(timezone.utc)

# GFS runs every 6 hours (00, 06, 12, 18 UTC)
# Subtractions account for the ~4 hour NOAA processing/upload lag
lagged_time = now_utc - timedelta(hours=4)
cycle_hour = (lagged_time.hour // 6) * 6

date_folder = lagged_time.strftime("%Y%m%d")
cycle_str = f"{cycle_hour:02d}"

# OpenSpace Filename & Timestamp: YYYY-MM-DDThh-mm-ss-nnn.json
run_date_iso = lagged_time.strftime("%Y-%m-%d")
output_filename = f"{run_date_iso}T{cycle_str}-00-00-000.json"
timestamp_iso = f"{run_date_iso}T{cycle_str}:00:00.000"

print(f"Current UTC Time:       {now_utc.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Fetching GFS Cycle:     {date_folder} at {cycle_str}:00 UTC")
print(f"OpenSpace Output File:  {output_filename}")

### Download NOAA GFS from AWS and Trace Fieldlines

In [ ]:
grib_url = f"https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.{date_folder}/{cycle_str}/atmos/gfs.t{cycle_str}z.pgrb2.0p25.f000"
local_grib_file = f"gfs.t{cycle_str}z.pgrb2.0p25.f000"

if not os.path.exists(local_grib_file):
    print(f"Downloading NOAA GFS file ({grib_url})...")
    urllib.request.urlretrieve(grib_url, local_grib_file)
    print("Download complete!")

print("Extracting 10m U and V wind vector fields...")
ds_u = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10u'})
ds_v = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10v'})

ds = xr.merge([ds_u, ds_v], compat='override')

print("Extraction complete!")
print("Extracting grid arrays for fast vector interpolation...")

fieldlines = trace_wind_fieldlines(ds)

print(f"Generated {len(fieldlines)} fieldlines")

### Export to Fieldlines JSON

In [ ]:
# Flatten all vertices across fieldlines into sequentially indexed keys ("0", "1", ...)
openspace_json_structure = {}
global_index = 0

for line in fieldlines:
    openspace_json_structure[str(global_index)] = {
        "time": timestamp_iso,  # Automatically matches the downloaded GFS cycle timestamp
        "trace": {
            "columns": [
                "x",
                "y",
                "z",
                "grid_value"
            ],
            "data": line
        }
    }
    global_index += 1

with open(output_filename, "w") as f:
    json.dump(openspace_json_structure, f, indent=2)

print(f"\nSaved {global_index} fieldlines with timestamp '{timestamp_iso}' to '{output_filename}'")

### JSON to OSFLS

In [ ]:
def convert_json_to_osfls(path):
    json_path = f"{path}.json"
    osfls_path = f"{path}.osfls"
    print(f"Loading JSON: {json_path}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    n_lines = len(data)
    first_key = list(data.keys())[0]

    # Extract timestamp and extra scalar variable names
    time_str = data[first_key]["time"]
    trigger_time_j2000 = iso_to_j2000_seconds(time_str)

    columns = data[first_key]["trace"]["columns"]
    extra_names = columns[3:]  # Columns after x, y, z (e.g., ['grid_value'])
    n_extras = len(extra_names)

    # Construct combined null-terminated variable names string
    all_names_bytes = b"".join([name.encode('utf-8') + b'\0' for name in extra_names])
    byte_size_all_names = len(all_names_bytes)

    # Flatten vertices and build line indexing arrays
    line_start = []
    line_count = []
    vertex_positions = []
    extra_quantities = [[] for _ in range(n_extras)]

    curr_index = 0
    for line_key, line_obj in data.items():
        pts = line_obj["trace"]["data"]
        pts_count = len(pts)

        line_start.append(curr_index)
        line_count.append(pts_count)
        curr_index += pts_count

        for pt in pts:
            # pt = [x, y, z, grid_value_1, ...]
            vertex_positions.extend(pt[:3])
            for i in range(n_extras):
                extra_quantities[i].append(pt[3 + i])

    n_points = len(line_start) and curr_index

    # Convert to standard C-types NumPy arrays
    line_start_arr = np.array(line_start, dtype=np.int32)
    line_count_arr = np.array(line_count, dtype=np.uint32)
    vertex_positions_arr = np.array(vertex_positions, dtype=np.float32)
    extra_quantities_arrs = [np.array(eq, dtype=np.float32) for eq in extra_quantities]

    print(f"Writing OSFLS Binary: {n_lines} lines, {n_points} total points...")
    with open(osfls_path, 'wb') as ofs:
        # Header Fields
        ofs.write(struct.pack('i', 0))                    # CurrentVersion = 0
        ofs.write(struct.pack('d', trigger_time_j2000))    # _triggerTime
        ofs.write(struct.pack('i', 0))                    # _model = Invalid/Generic (0)
        ofs.write(struct.pack('B', 0))                    # _isMorphable = 0 (false)
        ofs.write(struct.pack('Q', n_lines))              # nLines
        ofs.write(struct.pack('Q', n_points))             # nPoints
        ofs.write(struct.pack('Q', n_extras))             # nExtras
        ofs.write(struct.pack('Q', byte_size_all_names))  # byteSizeAllNames

        # Array Data Payload
        line_start_arr.tofile(ofs)
        line_count_arr.tofile(ofs)
        vertex_positions_arr.tofile(ofs)
        for eq_arr in extra_quantities_arrs:
            eq_arr.tofile(ofs)

        # String Data Payload
        ofs.write(all_names_bytes)

    print(f"Successfully converted to binary file: '{osfls_path}'")

# Example usage
convert_json_to_osfls("2026-09-25T00-00-00-000")

## NOAA GFS to OSFLS (+ 6-Hour Forecasting Loop)

### Forecasting Loop

This takes 15-20 minutes to complete.

In [ ]:
# Determine latest GFS cycle run (accounting for ~4hr processing delay)
now_utc = datetime.now(timezone.utc)
lagged_time = (now_utc - timedelta(hours=4)).replace(minute=0, second=0, microsecond=0)
cycle_hour = (lagged_time.hour // 6) * 6

date_folder = lagged_time.strftime("%Y%m%d")
cycle_str = f"{cycle_hour:02d}"

# Define model run base time
base_datetime = datetime.strptime(f"{date_folder} {cycle_str}", "%Y%m%d %H").replace(tzinfo=timezone.utc)

# Choose forecast hours to process (e.g., f000 to f006 for a 6-hour animation sequence)
FORECAST_HOURS = range(0, 7)  # 0 to 6

for f_hr in FORECAST_HOURS:
    f_str = f"f{f_hr:03d}"

    # Calculate timestamps
    valid_datetime = base_datetime + timedelta(hours=f_hr)
    trigger_time_j2000 = iso_to_j2000_seconds(valid_datetime)

    # OpenSpace sequence naming format: YYYY-MM-DDThh-mm-ss-nnn.osfls
    filename_osfls = valid_datetime.strftime("%Y-%m-%dT%H-%M-%S-000.osfls")

    print(f"Processing Forecast Hour: {f_str} ({valid_datetime.strftime('%Y-%m-%dT%H:%M:%S.000')})")

    # Download GRIB file
    grib_url = f"https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.{date_folder}/{cycle_str}/atmos/gfs.t{cycle_str}z.pgrb2.0p25.{f_str}"
    local_grib_file = f"gfs.t{cycle_str}z.pgrb2.0p25.{f_str}"

    if not os.path.exists(local_grib_file):
        print(f"Downloading {grib_url}...")
        urllib.request.urlretrieve(grib_url, local_grib_file)

    # Load 10m U/V components
    ds_u = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10u'})
    ds_v = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10v'})
    ds = xr.merge([ds_u, ds_v], compat='override')

    fieldlines = trace_wind_fieldlines(ds)

    # Write directly to OSFLS binary
    export_to_osfls(filename_osfls, fieldlines, trigger_time_j2000, scalar_name="grid_value")
    print(f"Successfully exported binary: {filename_osfls} ({len(fieldlines)} lines)")

    # Clean up local GRIB download to conserve disk space
    ds.close()
    ds_u.close()
    ds_v.close()
    if os.path.exists(local_grib_file):
        os.remove(local_grib_file)

    if os.path.exists(f"{local_grib_file}.47d85.idx"):
        os.remove(f"{local_grib_file}.47d85.idx")

### Historical Loop

In [ ]:
HOURS_LOOKBACK = 48  # Options: 24, 48, 168 (7 days)
STEP_INTERVAL_HOURS = 1   # Generates a frame for EVERY hour

now_utc = datetime.now(timezone.utc)
# Account for current run availability/upload lag (~4 hours)
end_time = (now_utc - timedelta(hours=4)).replace(minute=0, second=0, microsecond=0)
start_time = end_time - timedelta(hours=HOURS_LOOKBACK)

print(f"Generating hourly historical OSFLS sequence from {start_time} to {end_time}\n")

current_time = start_time
while current_time <= end_time:
    # 1. Determine base 6-hour cycle run (00, 06, 12, 18)
    cycle_hour = (current_time.hour // 6) * 6
    date_folder = current_time.strftime("%Y%m%d")
    cycle_str = f"{cycle_hour:02d}"

    # 2. Calculate forecast offset hour (f000, f001, ..., f005)
    f_hr = current_time.hour - cycle_hour
    f_str = f"f{f_hr:03d}"

    # 3. Calculate exact valid timestamp & OpenSpace J2000 trigger time
    trigger_time_j2000 = iso_to_j2000_seconds(current_time.strftime("%Y-%m-%dT%H:%M:%S.000"))
    filename_osfls = current_time.strftime("%Y-%m-%dT%H-%M-%S-000.osfls")

    print(f"Target Time: {current_time.strftime('%Y-%m-%d %H:00 UTC')} | Cycle: {date_folder} t{cycle_str}z | Offset: {f_str}")

    # 4. Construct S3 URL with forecast offset
    grib_url = f"https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.{date_folder}/{cycle_str}/atmos/gfs.t{cycle_str}z.pgrb2.0p25.{f_str}"
    local_grib_file = f"gfs.t{cycle_str}z.pgrb2.0p25.{f_str}"

    try:
        if not os.path.exists(local_grib_file):
            urllib.request.urlretrieve(grib_url, local_grib_file)

        # Load U/V components
        ds_u = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10u'})
        ds_v = xr.open_dataset(local_grib_file, engine='cfgrib', filter_by_keys={'shortName': '10v'})
        ds = xr.merge([ds_u, ds_v], compat='override')

        # Trace streamlines & write binary OSFLS
        fieldlines = trace_wind_fieldlines(ds)
        export_to_osfls(filename_osfls, fieldlines, trigger_time_j2000, scalar_name="grid_value")
        print(f"     Exported: {filename_osfls} ({len(fieldlines)} lines)\n")

        # Clean up memory & temp files
        ds.close()
        ds_u.close()
        ds_v.close()
        if os.path.exists(local_grib_file):
            os.remove(local_grib_file)

        if os.path.exists(f"{local_grib_file}.47d85.idx"):
            os.remove(f"{local_grib_file}.47d85.idx")

    except Exception as e:
        print(f"     Failed to fetch {grib_url}: {e}\n")

    # Advance by 1 hour
    current_time += timedelta(hours=STEP_INTERVAL_HOURS)

### Bulk Export from Colab

In [ ]:
try:
    import zipfile
    import glob
    from google.colab import files

    zip_filename = "openspace_osfls_sequence.zip"

    # Find all generated ISO osfls binary files
    osfls_files = glob.glob("*.osfls")

    print(f"Zipping {len(osfls_files)} OSFLS binary files...")
    with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file in osfls_files:
            zipf.write(file)
            print(f"  Added: {file}")

    print(f"\nDownloading {zip_filename}...")
    files.download(zip_filename)
except ImportError:
    print("Running in local environment: Skipping Google Colab browser download.")